In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet('/home/camarada/Documents/projects/temp-grss-nasa/data_/results_colab/monte_carlo_predictions_forAPRIL26-runon_01-05-26.parquet')

In [4]:
df.columns

Index(['model_type', 'ft_iteration', 'seed', 'lr', 'dropout_ft', 'val_loss',
       'conv_choice', 'trainable_param_percent', 'prediction_jan',
       'prediction_feb'],
      dtype='object')

In [10]:
mapper = {'prediction_jan':'mar_pred', 'prediction_fev':'april_pred'}

In [12]:
import numpy as np
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource, Div, HoverTool, Span, NumeralTickFormatter
)
from bokeh.io import push_notebook
from ipywidgets import interact, FloatSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

output_notebook()

## change name 
df = df.rename(columns=mapper)
df[mapper['prediction_jan']] = df[mapper['prediction_jan']].astype(float)
df[mapper['prediction_fev']] = df[mapper['prediction_fev']].astype(float)

def make_histogram(values, bins=14):
    counts, edges = np.histogram(values, bins=bins)
    return dict(
        top=counts,
        left=edges[:-1],
        right=edges[1:],
        count=counts,
        center=(edges[:-1] + edges[1:]) / 2,
    )

def compute_stats(values):
    return {
        'mean':   np.mean(values),
        'median': np.median(values),
        'std':    np.std(values),
        'p5':     np.percentile(values, 5),
        'p95':    np.percentile(values, 95),
        'n':      len(values),
    }

# ── extracted so update() can call it too ──────────────────────────────────
def make_stats_html(title, s, mean_color, bar_color):
    return f"""
    <div style="font-family:monospace; font-size:12px; color:#4a7090;
                margin-top:8px; line-height:1.8">
      <span style="color:#c8e8f8; font-size:13px">{title}</span><br>
      <b style="color:{mean_color}">μ</b> {s['mean']:.4f} &nbsp;
      <b style="color:#94b8d0">med</b> {s['median']:.4f} &nbsp;
      <b style="color:#94b8d0">σ</b> {s['std']:.4f} &nbsp;
      <b style="color:#94b8d0">P5</b> {s['p5']:.4f} &nbsp;
      <b style="color:#94b8d0">P95</b> {s['p95']:.4f} &nbsp;
      <b style="color:#94b8d0">n</b> {s['n']}
    </div>
    """

def build_dist_panel(values, title, bar_color, mean_color):
    hist = make_histogram(values)
    s    = compute_stats(values)
    src  = ColumnDataSource(hist)

    p = figure(
        title=title, height=300, width=520,
        toolbar_location=None,
        background_fill_color="#0a1520",
        border_fill_color="#060e16",
        outline_line_color="#0f2535",
    )
    p.title.text_color      = "#c8e8f8"
    p.title.text_font_size  = "14px"
    p.title.text_font_style = "italic"

    p.quad(
        top="top", bottom=0, left="left", right="right",
        source=src,
        fill_color=bar_color, fill_alpha=0.55,
        line_color="#060e16", line_width=0.8,
    )

    mean_span = Span(location=s['mean'],  dimension='height',
                     line_color=mean_color, line_dash='dashed', line_width=2)
    p5_span   = Span(location=s['p5'],    dimension='height',
                     line_color=bar_color, line_dash='dotted', line_width=1, line_alpha=0.5)
    p95_span  = Span(location=s['p95'],   dimension='height',
                     line_color=bar_color, line_dash='dotted', line_width=1, line_alpha=0.5)
    p.add_layout(mean_span)
    p.add_layout(p5_span)
    p.add_layout(p95_span)

    p.add_tools(HoverTool(tooltips=[
        ("Range", "@left{0.000} – @right{0.000}"),
        ("Runs",  "@count"),
    ]))

    for ax in [p.xaxis, p.yaxis]:
        ax.axis_label_text_color  = "#4a6a80"
        ax.major_label_text_color = "#3a5a70"
        ax.axis_line_color        = "#0f2535"
        ax.major_tick_line_color  = "#0f2535"
        ax.minor_tick_line_color  = None
    p.xgrid.grid_line_color = None
    p.ygrid.grid_line_color = "#0f2030"
    p.ygrid.grid_line_dash  = [4, 4]
    p.xaxis.formatter       = NumeralTickFormatter(format="0.00")
    p.xaxis.axis_label      = "Predicted Anomaly (°C)"
    p.yaxis.axis_label      = "# Runs"

    stats_div = Div(
        text=make_stats_html(title, s, mean_color, bar_color),
        width=520,
        styles={"background": "#0a1520", "padding": "6px 10px",
                "border": f"1px solid {bar_color}33", "border-radius": "0 0 8px 8px"},
    )

    # ✅ return stats_div so update() can write to it
    return column(p, stats_div), src, mean_span, p5_span, p95_span, stats_div


jan_panel, jan_src, jan_mean, jan_p5, jan_p95, jan_div = build_dist_panel(
    df[mapper['prediction_jan']].values, "March 2026", "#4fc3f7", "#00e5ff"
)
feb_panel, feb_src, feb_mean, feb_p5, feb_p95, feb_div = build_dist_panel(
    df[mapper['prediction_fev']].values, "April 2026",    "#f06292", "#ff80ab"
)


header = Div(text="""
<div style="font-family:'Georgia',serif; color:#c8e8f8; font-size:22px; padding:12px 0 4px">
  Global Temperature Anomaly &nbsp;
  <span style="color:#1e6080;font-size:14px">Monte Carlo Ensemble</span>
</div>
""", width=1080)

# 2. Build Layout
layout = column(header, row(jan_panel, feb_panel))

# 3. Show and Capture Handle
handle = show(layout, notebook_handle=True)


model_types = ["All"] + sorted(df['model_type'].unique().tolist())

def update(max_val_loss=0.1, model_type="All"):
    
    mask = df['val_loss'] <= max_val_loss
    if model_type != "All":
        mask &= df['model_type'] == model_type
    filtered = df[mask]
    print(f"Updating with {len(filtered)} rows")
    if len(filtered) == 0:
        print("⚠️  No runs match the current filter.")
        return

    panels = [
        (mapper['prediction_jan'], jan_src, jan_mean, jan_p5, jan_p95, jan_div, "March 2026", "#4fc3f7", "#00e5ff"),
        (mapper['prediction_fev'], feb_src, feb_mean, feb_p5, feb_p95, feb_div, "April 2026",    "#f06292", "#ff80ab"),
    ]

    for col_name, src, mean_span, p5_span, p95_span, stats_div, title, bar_color, mean_color in panels:
        vals = filtered[col_name].values
        s    = compute_stats(vals)

        # ✅ full reassignment — guarantees Bokeh's change detection fires
        src.data = dict(make_histogram(vals))

        mean_span.location = s['mean']
        p5_span.location   = s['p5']
        p95_span.location  = s['p95']

        # ✅ update the stats text too
        stats_div.text = make_stats_html(title, s, mean_color, bar_color)

    push_notebook(handle=handle)

interact(
    update,
    max_val_loss=FloatSlider(
        min=0.01, max=df['val_loss'].max() + 0.01,
        step=0.001, value=0.1,
        description="Val Loss ≤",
    ),
    model_type=Dropdown(options=model_types, value="All", description="Model"),
)

Loading BokehJS ...

interactive(children=(FloatSlider(value=0.1, description='Val Loss ≤', max=0.14143713772296906, min=0.01, step…

<function __main__.update(max_val_loss=0.1, model_type='All')>

In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from ipywidgets import interact, FloatSlider, Dropdown, VBox

# 1. Setup the FigureWidget with subplots
fig = go.FigureWidget(make_subplots(
    rows=1, cols=2, 
    subplot_titles=("March 2026", "April 2026"),
    horizontal_spacing=0.1
))

# 2. Add Histogram Traces
# Trace 0 = Feb, Trace 1 = Mar
fig.add_trace(go.Histogram(x=df[mapper['prediction_jan']], name="Mar", marker_color='#4fc3f7', opacity=0.6), row=1, col=1)
fig.add_trace(go.Histogram(x=df[mapper['prediction_fev']], name="April", marker_color='#f06292', opacity=0.6), row=1, col=2)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#060e16",
    plot_bgcolor="#0a1520",
    showlegend=False,
    height=450,
    margin=dict(l=50, r=50, t=80, b=50)
)

# Helper to create a vertical line dict
def create_v_line(val, color, dash, xref):
    return dict(
        type="line", x0=val, x1=val, y0=0, y1=1,
        yref="paper", xref=xref, # <--- CRITICAL: xref='x' for col 1, 'x2' for col 2
        line=dict(color=color, width=2, dash=dash)
    )

def update_plot(max_val_loss=0.1, model_type="All"):
    mask = df['val_loss'] <= max_val_loss
    if model_type != "All":
        mask &= df['model_type'] == model_type
    
    filtered = df[mask]
    if filtered.empty: return

    with fig.batch_update():
        # Update Data
        fig.data[0].x = filtered[mapper['prediction_jan']]
        fig.data[1].x = filtered[mapper['prediction_fev']]
        
        # Stats
        f_mean, f_med = filtered[mapper['prediction_jan']].mean(), filtered[mapper['prediction_jan']].median()
        m_mean, m_med = filtered[mapper['prediction_fev']].mean(), filtered[mapper['prediction_fev']].median()
        
        # Update Shapes (Lines)
        # 'x' refers to the first subplot x-axis, 'x2' to the second
        fig.layout.shapes = [
            # FEB Lines (xref="x")
            create_v_line(f_mean, "#00e5ff", "dash", "x"),   # Mean
            create_v_line(f_med,  "#ffffff", "dot",  "x"),   # Median
            
            # MAR Lines (xref="x2")
            create_v_line(m_mean, "#ff80ab", "dash", "x2"),  # Mean
            create_v_line(m_med,  "#ffffff", "dot",  "x2")   # Median
        ]
        
        # Optional: Add annotations to clarify which line is which
        fig.layout.annotations[0].text = f"Mar: μ={f_mean:.3f}, med={f_med:.3f}"
        fig.layout.annotations[1].text = f"April: μ={m_mean:.3f}, med={m_med:.3f}"

# 5. Controller UI
model_types = ["All"] + sorted(df['model_type'].unique().tolist())
interact(update_plot, 
         max_val_loss=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1),
         model_type=Dropdown(options=model_types))

display(fig)

interactive(children=(FloatSlider(value=0.1, description='max_val_loss', max=0.5, min=0.01, step=0.01), Dropdo…

FigureWidget({
    'data': [{'marker': {'color': '#4fc3f7'},
              'name': 'Mar',
              'opacity': 0.6,
              'type': 'histogram',
              'uid': '3a144255-83d6-4b3d-be91-c030f41770c0',
              'x': {'bdata': ('AAAAAJyz9j8AAABAFQH0PwAAAGCKkv' ... 'AAIGTi9D8AAABAPc71PwAAAED2yfQ/'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'},
             {'marker': {'color': '#f06292'},
              'name': 'April',
              'opacity': 0.6,
              'type': 'histogram',
              'uid': '7e8a9c3e-3c76-4f8c-b598-f32a4dfbcd64',
              'x': {'bdata': ('AAAA4BON8j8AAABg/cryPwAAAADuSf' ... 'AA4LLu8z8AAACAcif0PwAAAECBbPM/'),
                    'dtype': 'f8'},
              'xaxis': 'x2',
              'yaxis': 'y2'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Mar: μ=1.296, med=1.293',
               

In [14]:
df.select_dtypes(include="float64").describe()

,lr,dropout_ft,val_loss,trainable_param_percent,mar_pred,april_pred
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.000411,0.304800,0.036627,38.291519,1.237936,1.164639
std,0.000361,0.137458,0.034882,0.001511,0.156375,0.131853
min,0.000010,0.100000,0.009979,38.290010,0.885572,0.902883
25%,0.000100,0.200000,0.018135,38.290010,1.220895,1.116968
50%,0.000400,0.300000,0.021036,38.291519,1.278164,1.197239
75%,0.000500,0.400000,0.026096,38.293028,1.332160,1.256040
max,0.001000,0.500000,0.131437,38.293028,1.564268,1.485993


In [16]:
df.loc[df['model_type']=='YearInjection'].select_dtypes(include="float64").describe()

,lr,dropout_ft,val_loss,trainable_param_percent,fev_pred,mar_pred
count,250.000000,250.000000,250.000000,2.500000e+02,250.000000,250.000000
mean,0.000411,0.304800,0.028705,3.829303e+01,1.240473,1.311854
std,0.000361,0.137596,0.015454,4.271809e-14,0.099369,0.117206
min,0.000010,0.100000,0.010814,3.829303e+01,1.095707,1.114300
25%,0.000100,0.200000,0.018590,3.829303e+01,1.157757,1.261342
50%,0.000400,0.300000,0.022462,3.829303e+01,1.248550,1.313783
75%,0.000500,0.400000,0.028697,3.829303e+01,1.314488,1.390478
max,0.001000,0.500000,0.061663,3.829303e+01,1.531287,1.644781
